In [1]:
import os
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, RocCurveDisplay, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import plotly.express as px
import matplotlib.pyplot as plt
import bioframe as bf

pd.options.display.max_columns = 100

# Functions

In [2]:
def training(df, features, test_chrom=None, test_sample=None):
    """ Trains a model returns classifications. """

    if test_chrom != None:
        X_train = df.loc[~df['chrom'].isin(test_chrom), features]
        X_test = df.loc[df['chrom'].isin(test_chrom), features]
        y_train = df.loc[~df['chrom'].isin(test_chrom), 'confirmed']
        y_test = df.loc[df['chrom'].isin(test_chrom), 'confirmed']
    elif test_sample != None:
        X_train = df.loc[~df['sample'].isin(test_sample), features]
        X_test = df.loc[df['sample'].isin(test_sample), features]
        y_train = df.loc[~df['sample'].isin(test_sample), 'confirmed']
        y_test = df.loc[df['sample'].isin(test_sample), 'confirmed']
    else:
        return None

    RF = RandomForestClassifier(n_estimators=100)
    RF.fit(X_train, y_train)

    predictions = RF.predict(X_test)
    proba = RF.predict_proba(X_test)
    X_test['confirmed'] = y_test
    X_test['pred_dicast'] = predictions
    X_test['qual_dicast'] = proba[:, 1]
    result = df[['id', 'sample', 'method', 'type', 'chrom', 'start', 'end', 'filter', 'qual']].merge(X_test[['confirmed', 'pred_dicast', 'qual_dicast']], left_index=True, right_index=True)

    return result

In [3]:
def load_sample_data(SAMPLE, REF):
    """ Load sample data from a single sample. """
    
    FEATURE_DIR = f'/confidential/FamilyR13/DATA/10x/sv_compare/results/{SAMPLE}_{REF}/ensemble'
    df_raw = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.raw.tsv', sep='\t')
    df_ref = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.ref.tsv', sep='\t', low_memory=False)

    filenames_aln_ill = glob(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.aln.ill.*.tsv')
    df_aln_ill = pd.concat([pd.read_csv(f, sep='\t') for f in filenames_aln_ill], ignore_index=True)

    df = df_raw.merge(df_ref.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='inner')
    df = df.merge(df_aln_ill.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='inner')

    return df

In [4]:
# Mark variants which have been found by multiple callers

def extract_overlap_ids(df1, df2):
    """ Extracts SV IDs of overlapping variants"""

    closest_intervals = bf.closest(df1, df2, suffixes=('_1','_2'))
    closest_intervals = closest_intervals.dropna(subset=['id_1', 'id_2']).reset_index(drop=True)
    closest_intervals['diff_start'] = abs(closest_intervals['start_1'] - closest_intervals['start_2'])
    closest_intervals['diff_end'] = abs(closest_intervals['end_1'] - closest_intervals['end_2'])
    closest_intervals['diff_size'] = closest_intervals.apply(lambda x: min([x['size_1'], x['size_2']]) / max([x['size_1'], x['size_2']]), axis=1)
    overlapping_svs = closest_intervals[(closest_intervals['diff_start'] < 50) & (closest_intervals['diff_end'] < 50) & (closest_intervals['diff_size'] > 0.7)].copy()
    
    return overlapping_svs[['id_1', 'id_2']].reset_index(drop=True)

In [5]:
def merge_overlapping_svs(result, method_dfs, sample, methods):
    result_sample = result[result['sample'] == sample].copy().reset_index(drop=True)
    for i in range(len(methods)):
        for j in range(i+1, len(methods)):
            method_df_sample_1 = method_dfs[methods[i]][method_dfs[methods[i]]['sample'] == sample].copy().reset_index(drop=True)
            method_df_sample_2 = method_dfs[methods[j]][method_dfs[methods[j]]['sample'] == sample].copy().reset_index(drop=True)
            overlap_ids = extract_overlap_ids(method_df_sample_1, method_df_sample_2)
            for k in range(len(overlap_ids)):
                result_sample.loc[result_sample['id'] == overlap_ids['id_1'][k], 'qual_' + methods[j]] = result_sample.loc[result_sample['id'] == overlap_ids['id_2'][k], 'qual_' + methods[j]].values
                result_sample.loc[result_sample['id'] == overlap_ids['id_2'][k], 'qual_' + methods[i]] = result_sample.loc[result_sample['id'] == overlap_ids['id_1'][k], 'qual_' + methods[i]].values
            
    # Normalize Qualities
    for method in methods:
        result_sample['qual_' + method] = result_sample['qual_' + method] / result_sample['qual_' + method].max()

    return result_sample

In [6]:
def check_caller_support(row, min_num_callers):
    caller_count = 0
    for qual in row:
        if qual != 0:
            caller_count += 1
    if caller_count >= min_num_callers:
        return row.sum()
    else:
        return 0

# Script

In [7]:
# PARAMETERS
DATE = '20230123'
SAMPLES = ['17-08', '176-98', '146-97']
REF = 'hg38'
TYPE = 'DUP'
CHROMS = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 
          'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX']

In [19]:
# Load Data
dfs = []
for SAMPLE in SAMPLES:
    df = load_sample_data(SAMPLE, REF)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [20]:
df = df[:10].copy().reset_index(drop=True)

In [23]:
np.log2(1000 / 50)

4.321928094887363

In [27]:
df.drop([0, 2, 4])

,id,sample,tech,method,type,chrom,start,chrom2,end,size,filter,qual,confirmed,rep_LINE,rep_SINE,rep_LTR,rep_DNA,rep_Simple_repeat,rep_Satellite,rep_Low_complexity,rep_Retroposon,rep_snRNA,rep_tRNA,rep_srpRNA,rep_rRNA,rep_RC,rep_scRNA,rep_RNA,rep_VNTR,rep_STR,cpg_islands,centromeres,asmb_gaps,alt_haps,GC_content_left,GC_content_right,ill_cov_mean_I,ill_cov_mean_II,ill_cov_mean_III,ill_cov_mean_IV,ill_cov_std_I,ill_cov_std_II,ill_cov_std_III,ill_cov_std_IV,ill_isize_mean_I,ill_isize_mean_II,ill_isize_mean_III,ill_isize_mean_IV,ill_isize_std_I,ill_isize_std_II,ill_isize_std_III,ill_isize_std_IV,ill_mapq_mean_I,ill_mapq_mean_II,ill_mapq_mean_III,ill_mapq_mean_IV,ill_mapq_std_I,ill_mapq_std_II,ill_mapq_std_III,ill_mapq_std_IV,ill_clipreads_I,ill_clipreads_II,ill_clipreads_III,ill_clipreads_IV,ill_splitreads_I,ill_splitreads_II,ill_splitreads_III,ill_splitreads_IV,ill_disco_ff_I,ill_disco_ff_II,ill_disco_ff_III,ill_disco_ff_IV,ill_disco_rr_I,ill_disco_rr_II,ill_disco_rr_III,ill_disco_rr_IV,test
3,DUP00000003,17-08,ILL,delly,DUP,chr1,136586,NaN,136974,388.0,LowQual,42.0,0,5791.0,2212.0,3810.0,2733.0,2752,44024.0,25207.0,700124.0,20809.0,492524,1268471.0,1746694,3612172.0,7845903.0,9810344.0,769899.0,1615.0,1023.0,121889485.0,70692.0,2311836.0,68.0,72.0,-0.886,-0.221,0.265,-0.236,-1.620,-1.624,-1.551,-0.921,0.066,-0.101,-0.461,-0.369,1.393,1.457,1.376,1.105,-1.223,-1.438,-2.051,-2.027,0.089,0.144,-0.189,-0.100,0.174,0.250,0.181,0.180,0.000,0.015,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.015,0.011,0.000,NaN
5,DUP00000005,17-08,ILL,delly,DUP,chr1,668871,NaN,669162,291.0,LowQual,54.0,0,0.0,973.0,2748.0,17351.0,848,220464.0,1389.0,167936.0,89070.0,36186,736283.0,1214506,3079984.0,7313715.0,9278156.0,237711.0,1422.0,57602.0,121357297.0,82883.0,1779648.0,58.2,58.0,-1.088,-0.855,0.008,-0.767,-3.084,-3.846,-0.975,-1.759,-0.223,-0.323,-0.344,-0.271,0.974,1.157,1.298,1.232,-1.600,-1.271,-3.115,-3.736,-0.213,-0.040,-0.907,-1.312,0.293,0.116,0.247,0.362,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.054,0.049,0.000,0.017,NaN
6,DEL00000006,17-08,ILL,delly,DEL,chr1,789483,NaN,224012409,223222926.0,PASS,185.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41.4,38.0,-0.079,-1.229,0.576,0.617,-0.600,-0.564,-0.067,0.103,13.617,15.594,11.839,13.977,19.041,19.976,18.164,19.216,-0.466,-0.422,-0.538,-1.090,0.493,0.469,0.568,0.584,0.179,0.392,0.156,0.547,0.333,0.344,0.188,0.476,0.119,0.062,0.049,0.148,0.048,0.109,0.062,0.100,NaN
7,DUP00000007,17-08,ILL,delly,DUP,chr1,789491,NaN,224014604,223225113.0,PASS,1582.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,41.0,37.6,-0.222,-1.379,0.907,1.379,-0.841,-0.864,1.238,0.475,13.687,15.934,14.537,10.771,19.075,20.126,19.485,17.633,-0.484,-0.341,-0.448,-1.061,0.498,0.406,0.326,0.399,0.226,0.438,0.536,0.468,0.350,0.305,0.455,0.556,0.125,0.068,0.117,0.136,0.050,0.102,0.140,0.189,NaN
8,DUP00000008,17-08,ILL,delly,DUP,chr1,790133,NaN,224013400,223223267.0,LowQual,50.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,45.4,42.0,0.105,2.331,1.640,1.488,-0.150,2.818,-0.417,-0.541,12.987,15.075,13.049,11.091,18.732,19.686,18.763,17.792,-0.838,-1.764,-0.507,-0.275,0.532,0.083,0.525,0.350,0.494,0.512,0.172,0.197,0.246,0.317,0.257,0.190,0.062,0.127,0.084,0.054,0.100,0.108,0.056,0.033,NaN
9,DUP00000009,17-08,ILL,delly,DUP,chr1,790161,NaN,125071125,124280964.0,LowQual,54.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.8,36.0,1.309,2.989,1.211,1.262,2.227,1.895,1.158,-0.470,13.508,15.023,13.892,2.871,18.988,19.670,18.735,7.858,-1.533,-1.879,-0.539,-0.485,0.279,0.026,0.254,0.321,0.600,0.266,0.332,0.221,0.397,0.323,0.040,0.214,0.104,0.126,0.115,0.077,0.117,0.106,0.138,0.077,NaN


In [9]:
# Filter NaNs
features = ['size'] + list(df.columns[13:])
df = df.dropna(subset=features).copy().reset_index(drop=True)

# Select SV Type
df = df[df['type'] == TYPE].copy().reset_index(drop=True)

## Create set for manual curation

In [10]:
methods = list(df['method'].unique())
for chrom in tqdm(CHROMS):
    result = training(df, features, test_chrom=[chrom])
    curr_fp_svs = list(result.loc[(result['confirmed'] == 0) & (result['pred_dicast'] == 1), 'id'])
    curr_fn_svs = list(result.loc[(result['confirmed'] == 1) & (result['pred_dicast'] == 0), 'id'])

    method_dfs = dict()
    methods = list(result['method'].unique())
    for method in methods:
        result['qual_' + method] = 0
        result.loc[result['method'] == method, 'qual_' + method] = result.loc[result['method'] == method, 'qual']
        method_dfs[method] = df.loc[df['method'] == method, ['id', 'sample', 'method', 'chrom', 'start', 'end', 'type', 'size']].copy()
    result.drop('qual', axis=1, inplace=True)

    result = pd.concat([merge_overlapping_svs(result, method_dfs, sample, methods) for sample in SAMPLES], ignore_index=True)
    result['qual_one caller support'] = result[['qual_' + method for method in methods]].sum(axis=1)
    result['qual_one caller support'] = result['qual_one caller support'].apply(lambda x: np.round(x, 2))
    for method in methods:
        result['qual_' + method] = result['qual_' + method].apply(lambda x: np.round(x, 2))

    # ID needs to be fourth column for Jakobs webapp
    result = result[list(result.columns[1:4]) + ['id'] + list(result.columns[4:])]

    result = result[(result['confirmed'] == 0) & ((result['qual_dicast'] > 0.4) | (result['qual_one caller support'] > 0.4))]
    for sample in SAMPLES:
        if not os.path.isdir(f'/confidential/tGenVar/sv_manual_curation/{sample}/{TYPE}/{chrom}/images'):
            os.makedirs(f'/confidential/tGenVar/sv_manual_curation/{sample}/{TYPE}/{chrom}/images')
        result_sample = result[result['sample'] == sample].copy().reset_index(drop=True)
        result_sample.to_csv(f'/confidential/tGenVar/sv_manual_curation/{sample}/{TYPE}/{chrom}/{DATE}_{sample}_{TYPE}_{chrom}.tsv', index=False, sep='\t')

100%|██████████| 23/23 [08:39<00:00, 22.57s/it]


# Evaluate Classifier

In [152]:
chrom_train = ['chr1', 'chr3', 'chr4', 'chr5', 'chr7', 'chr8', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr19', 'chr20', 'chr21']
chrom_test = ['chr2', 'chr6', 'chr9', 'chr18', 'chr22']
sample_test = ['176-98']

In [153]:
result = training(df, features, test_sample=sample_test)

In [154]:
method_dfs = dict()
methods = list(result['method'].unique())
for method in methods:
    result['qual_' + method] = 0
    result.loc[result['method'] == method, 'qual_' + method] = result.loc[result['method'] == method, 'qual']
    method_dfs[method] = df.loc[df['method'] == method, ['id', 'sample', 'method', 'chrom', 'start', 'end', 'type', 'size']].copy()
result.drop('qual', axis=1, inplace=True)

In [155]:
result = pd.concat([merge_overlapping_svs(result, method_dfs, sample, methods) for sample in SAMPLES], ignore_index=True)

In [156]:
result['qual_one caller support'] = result[['qual_' + method for method in methods]].sum(axis=1)
result['qual_two caller support'] = result.apply(lambda x: check_caller_support(x[['qual_' + method for method in methods]], 2), axis=1)
result['qual_three caller support'] = result.apply(lambda x: check_caller_support(x[['qual_' + method for method in methods]], 3), axis=1)

In [157]:
methods_extended = ['dicast'] + methods + ['one caller support', 'two caller support', 'three caller support']
pr_rc_dict = {'method' : [], 'precision' : [], 'recall' : []}
for method in methods_extended:
    precision, recall, _ = precision_recall_curve(result['confirmed'], result['qual_' + method])
    pr_rc_dict['method'].extend([method] * (len(precision) - 1))
    pr_rc_dict['precision'].extend(precision[1:])
    pr_rc_dict['recall'].extend(recall[1:])
pr_rc_df = pd.DataFrame(pr_rc_dict)

In [158]:
colors = ['black', '#1f77b4', '#ff7f0e', 'darkred', '#1f77b4', '#ff7f0e', 'darkred']
dash = ['solid', 'solid', 'solid', 'dot', 'dot', 'dot', 'dot']
circle_bg_white = [0, 0, 0, 1, 1, 1, 1]
fig = px.line(x='recall', y='precision', color='method', 
              data_frame=pr_rc_df, 
              title='Precision-Recall Curve without Manual Curation (' + TYPE + ')', line_dash='method', 
              line_dash_sequence=dash,
              color_discrete_sequence=colors)

for i, method in enumerate(methods_extended):
    x = pr_rc_df[pr_rc_df['method'] == method].reset_index(drop=True).loc[0, 'recall']
    y = pr_rc_df[pr_rc_df['method'] == method].reset_index(drop=True).loc[0, 'precision']
    if circle_bg_white[i] == 1:
        fig.add_shape(type='circle', xref='x', yref='y', x0=x-0.005, y0=y-0.015, x1=x+0.005, y1=y+0.005, line_color=colors[i], line_width=2, opacity=1, fillcolor='white')
    else:
        fig.add_shape(type='circle', xref='x', yref='y', x0=x-0.005, y0=y-0.015, x1=x+0.005, y1=y+0.005, line_color=colors[i], line_width=2, opacity=1, fillcolor=colors[i])

fig.update_layout(plot_bgcolor='white', xaxis_title='Recall', yaxis_title='Precision', xaxis_linecolor='black', yaxis_linecolor='black')
fig.update_traces(line=dict(width=2))
fig.update_xaxes(ticks='outside', tickcolor='black', tickwidth=1, ticklen=5, gridcolor='lightgray', gridwidth=0.5, range=[0, 1.1])
fig.update_yaxes(ticks='outside', tickcolor='black', tickwidth=1, ticklen=5, gridcolor='lightgray', gridwidth=0.5, range=[0, 1.1])
fig.write_image('figures/pr_curve_no_curation_' + TYPE + '.png', width=1200, height=650, scale=3)